In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mplsoccer import Pitch

In [ ]:
ACTIONS_PATH = Path(
    r"D:\UIUC\Projects\World Cup 2026\Learning\processed_data\actions.parquet"
)

MATCH_METADATA_PATH = Path(
    r"D:\UIUC\Projects\World Cup 2026\Learning\processed_data\match_metadata.parquet"
)

XT_MODEL_PATH = Path(
    r"D:\UIUC\Projects\World Cup 2026\Learning\processed_data\general_xt_model.npy"
)
actions = pd.read_parquet(ACTIONS_PATH)

match_metadata = pd.read_parquet(
    MATCH_METADATA_PATH
)

xt_grid = np.load(XT_MODEL_PATH)
print("Actions shape:", actions.shape)
print("Match metadata shape:", match_metadata.shape)
print("xT model shape:", xt_grid.shape)

display(actions.head())
display(match_metadata.head())

In [ ]:
actions["match_id"] = (
    actions["match_id"].astype(int)
)

match_metadata["match_id"] = (
    match_metadata["match_id"].astype(int)
)
match_metadata = (
    match_metadata
    .drop_duplicates(subset="match_id")
    .copy()
)
actions_with_metadata = actions.merge(
    match_metadata,
    on="match_id",
    how="left",
    validate="many_to_one",
)
print(
    actions_with_metadata[
        ["competition_name", "season_name"]
    ].isna().sum()
)

In [ ]:
barcelona_seasons = (
    actions_with_metadata.loc[
        actions_with_metadata["team"]
        .str.contains(
            "Barcelona",
            case=False,
            na=False,
        ),
        [
            "competition_name",
            "season_name",
        ],
    ]
    .drop_duplicates()
    .sort_values(
        ["competition_name", "season_name"]
    )
)

display(barcelona_seasons)

In [ ]:
TARGET_TEAM = "Barcelona"
TARGET_COMPETITION = "La Liga"
TARGET_SEASON = "2015/2016"

team_actions = actions_with_metadata[
    (actions_with_metadata["team"] == TARGET_TEAM)
    & (
        actions_with_metadata["competition_name"]
        == TARGET_COMPETITION
    )
    & (
        actions_with_metadata["season_name"]
        == TARGET_SEASON
    )
].copy()
print("Team actions:", len(team_actions))

print(
    "Matches:",
    team_actions["match_id"].nunique(),
)

print(
    team_actions["event_type"].value_counts()
)